In [ ]:
# Usar la base de datos isabela_db
spark.sql("USE isabela_db")

# Pregunta 1: Top 10 películas mejor calificadas (rating promedio)
df_top10 = spark.sql("""
    SELECT 
        r.movieId,
        t.title,
        AVG(r.rating) AS avg_rating,
        COUNT(*) AS num_ratings
    FROM ratings r
    JOIN tmdb t ON r.movieId = t.tmdbId
    GROUP BY r.movieId, t.title
    ORDER BY avg_rating DESC
    LIMIT 10
""")
print("Pregunta 1 - Top 10 mejor calificadas:")
df_top10.show(truncate=False)

# Pregunta 2: Distribución de calificaciones (frecuencia por estrella entera)
df_distribution = spark.sql("""
    SELECT 
        ROUND(rating) AS star_rating,
        COUNT(*) AS frequency
    FROM ratings
    GROUP BY ROUND(rating)
    ORDER BY star_rating
""")
print("\nPregunta 2 - Distribución de ratings:")
df_distribution.show()

# Pregunta 3: Películas más populares (mayor número de ratings)
df_popular = spark.sql("""
    SELECT 
        movieId,
        COUNT(*) AS num_ratings
    FROM ratings
    GROUP BY movieId
    ORDER BY num_ratings DESC
    LIMIT 10
""")
print("\nPregunta 3 - Top 10 películas más populares (por cantidad de ratings):")
df_popular.show()

# Pregunta 4: Evolución temporal del rating promedio por mes (usando timestamp Unix)
# Nota: el timestamp está en segundos, lo convertimos a fecha y luego extraemos año-mes
df_temporal = spark.sql("""
    SELECT 
        YEAR(FROM_UNIXTIME(CAST(timestamp AS BIGINT))) AS year,
        MONTH(FROM_UNIXTIME(CAST(timestamp AS BIGINT))) AS month,
        AVG(rating) AS avg_rating,
        COUNT(*) AS total_ratings
    FROM ratings
    WHERE timestamp IS NOT NULL
    GROUP BY 
        YEAR(FROM_UNIXTIME(CAST(timestamp AS BIGINT))),
        MONTH(FROM_UNIXTIME(CAST(timestamp AS BIGINT)))
    ORDER BY year, month
""")
print("\nPregunta 4 - Evolución temporal del rating promedio por mes:")
df_temporal.show(20)  # Mostrar varios registros

# Pregunta 5: Relación entre presupuesto y rating promedio (top 20 presupuesto más alto)
df_budget_rating = spark.sql("""
    SELECT 
        t.tmdbId,
        t.title,
        t.budget,
        AVG(r.rating) AS avg_rating,
        COUNT(r.rating) AS num_ratings
    FROM tmdb t
    JOIN ratings r ON t.tmdbId = r.movieId
    WHERE t.budget > 0
    GROUP BY t.tmdbId, t.title, t.budget
    ORDER BY t.budget DESC
    LIMIT 20
""")
print("\nPregunta 5 - Top 20 películas con mayor presupuesto y su rating promedio:")
df_budget_rating.show(truncate=False)

# Opcional: calcular coeficiente de correlación entre presupuesto y rating promedio (solo para las que tienen ratings)
from pyspark.sql.functions import corr
corr_df = spark.sql("""
    SELECT 
        t.budget,
        AVG(r.rating) AS avg_rating
    FROM tmdb t
    JOIN ratings r ON t.tmdbId = r.movieId
    WHERE t.budget > 0
    GROUP BY t.tmdbId, t.budget
""")
correlation = corr_df.select(corr("budget", "avg_rating")).collect()[0][0]
print(f"\nCoeficiente de correlación entre presupuesto y rating promedio (Pearson): {correlation:.4f}")